# 11 — LangGraph Agents: Building Atlas

## What Is Atlas?

Atlas is ServiceTitan's AI sidekick announced at Pantheon 2025. It's a **tool-using LLM
agent** embedded directly in the contractor's workflow. Instead of navigating menus,
a dispatcher can say:

> "Assign the next available HVAC tech to the Smith job and send them the details."
> "Show me all jobs that are past their scheduled window."
> "Run a report of this week's revenue by job type."

Atlas executes these actions — it doesn't just answer questions. That's what makes it
an **agent** rather than a chatbot: it calls real functions that mutate real state.

## Why LangGraph? Agent Frameworks Compared

| Framework | Model | Best For | Weakness |
|---|---|---|---|
| **Simple LLM + prompt** | Stateless | Single-turn Q&A | Can't handle multi-step workflows |
| **LangChain AgentExecutor** | Chain-based | Simple tool use | Hard to control flow; opaque loops |
| **LangGraph (chosen)** | Explicit state machine | Multi-step, conditional, cyclable flows | More setup overhead |
| **AutoGen / CrewAI** | Multi-agent | Multiple cooperating agents | Overkill for single-agent dispatch |

**Why LangGraph specifically:**
- The state is explicit and typed (`TypedDict`) — every node receives and returns a
  well-defined state object. No hidden variables.
- Conditional edges let you route to different nodes based on state — critical for
  the confirm-before-execute safety pattern.
- The graph can pause mid-execution (at a write operation) and wait for human confirmation.
  LangChain's AgentExecutor has no clean way to do this.
- Production Atlas almost certainly uses a state machine because write operations
  (dispatching a tech, modifying a job, triggering a payment) require an audit trail
  and human approval gates.

## The Core Design: Read vs. Write Classification

The most important architectural decision in Atlas is splitting tools into two categories:

```
READ tools  → execute immediately (no side effects, safe to retry)
WRITE tools → pause graph, request confirmation, then execute

Examples:
  READ:  get_pending_jobs, get_available_techs, get_schedule_summary, run_report
  WRITE: assign_tech, create_job, reschedule_job, trigger_campaign, process_payment
```

This mirrors how production systems handle command/query separation (CQRS pattern).
It also means the LLM's routing decision is a classification task — not freeform generation —
which makes it much more reliable and auditable.

## The Full Flow

```
User message
    ↓
[parse node] → LLM intent extraction → {tool, args, is_write}
    ↓
[route] → READ  → [read_tool node] → execute immediately → [fmt] → response
        → WRITE → [stage_write node] → pause, show confirmation prompt → END
        → NONE  → [fmt] → direct LLM response
    
On confirmation:
User confirms/cancels
    ↓
[exec_confirmed node] → execute write tool → [fmt] → response
```


In [1]:
from typing import Literal, Optional, TypedDict
from langgraph.graph import StateGraph, END
import json, time, re
import langgraph; print('LangGraph installed OK')

LangGraph installed OK


## 1. Tool Registry

In [2]:
JOBS = [
    {'id':'J001','customer':'Smith HVAC',    'status':'pending',  'tech':None,'value':850},
    {'id':'J002','customer':'Jones Plumbing','status':'pending',  'tech':None,'value':1200},
    {'id':'J003','customer':'Brown Electric','status':'completed','tech':'T01','value':650},
]
TECHS = [
    {'id':'T01','name':'Carlos M.','available':False,'skill':'HVAC',   'jobs':3},
    {'id':'T02','name':'Priya S.', 'available':True, 'skill':'Plumbing','jobs':1},
    {'id':'T03','name':'James K.','available':True, 'skill':'HVAC',   'jobs':2},
]

def get_pending_jobs():     return [j for j in JOBS  if j['status']=='pending']
def get_available_techs():  return [t for t in TECHS if t['available']]
def get_schedule_summary():
    p = sum(1 for j in JOBS if j['status']=='pending')
    a = sum(1 for t in TECHS if t['available'])
    return {'pending_jobs':p,'available_techs':a,'utilization':f'{(3-a)/3*100:.0f}%'}

def assign_tech(job_id, tech_id):
    for j in JOBS:
        if j['id']==job_id:
            j['tech'],j['status'] = tech_id,'assigned'
            name = next(t['name'] for t in TECHS if t['id']==tech_id)
            return f'Assigned {name} to {job_id} ({j["customer"]})'
    return f'Job {job_id} not found'

TOOLS = {
    'get_pending_jobs':     (get_pending_jobs,     'read'),
    'get_available_techs':  (get_available_techs,  'read'),
    'get_schedule_summary': (get_schedule_summary, 'read'),
    'assign_tech':          (assign_tech,           'write'),
}
print('Read: ', [k for k,v in TOOLS.items() if v[1]=='read'])
print('Write:', [k for k,v in TOOLS.items() if v[1]=='write'])

Read:  ['get_pending_jobs', 'get_available_techs', 'get_schedule_summary']
Write: ['assign_tech']


## 2. Mock LLM (Function Calling Pattern)

In [3]:
def mock_llm(msg):
    # In production: Azure OpenAI with function_calling schema
    m = msg.lower()
    if 'pending' in m or 'unassigned' in m:
        return {'tool':'get_pending_jobs','args':{}}
    if 'available' in m:
        return {'tool':'get_available_techs','args':{}}
    if 'schedule' in m or 'summary' in m or 'overview' in m:
        return {'tool':'get_schedule_summary','args':{}}
    if 'assign' in m or 'dispatch' in m:
        jm = re.search(r'J\d{3}', msg.upper())
        tm = re.search(r'T\d{2}', msg.upper())
        return {'tool':'assign_tech',
                'args':{'job_id':  jm.group() if jm else 'J001',
                        'tech_id': tm.group() if tm else 'T02'}}
    return {'tool':None,'args':{}}

for q in ['Show pending jobs','Who is available?','Assign T02 to J001']:
    print(f'  {q!r} -> {mock_llm(q)}')

  'Show pending jobs' -> {'tool': 'get_pending_jobs', 'args': {}}
  'Who is available?' -> {'tool': 'get_available_techs', 'args': {}}
  'Assign T02 to J001' -> {'tool': 'assign_tech', 'args': {'job_id': 'J001', 'tech_id': 'T02'}}


## 3. Agent State + Nodes

In [4]:
class S(TypedDict):
    msg:     str
    intent:  Optional[dict]
    result:  Optional[object]
    pending: Optional[dict]
    confirm: Optional[bool]
    answer:  Optional[str]
    trace:   list

def parse(s):
    return {'intent':mock_llm(s['msg']),'trace':s['trace']+['parse']}

def read_tool(s):
    fn,_ = TOOLS[s['intent']['tool']]
    a = s['intent']['args']
    return {'result': fn(**a) if a else fn(),'trace':s['trace']+['read']}

def stage_write(s):
    t,a = s['intent']['tool'], s['intent']['args']
    print(f'  CONFIRM REQUIRED: {t}({a})')
    return {'pending':{'tool':t,'args':a},'trace':s['trace']+['stage']}

def exec_confirmed(s):
    if not s.get('confirm'):
        return {'answer':'Action cancelled.','trace':s['trace']+['cancel']}
    c = s['pending']
    fn,_ = TOOLS[c['tool']]
    return {'result':fn(**c['args']),'trace':s['trace']+['write']}

def fmt(s):
    r = s.get('result')
    if r is None:         ans = 'I help with scheduling and dispatch.'
    elif isinstance(r,list): ans = f'{len(r)} results: '+'; '.join(str(x) for x in r)
    elif isinstance(r,dict): ans = ' | '.join(f'{k}:{v}' for k,v in r.items())
    else:                 ans = str(r)
    return {'answer':ans,'trace':s['trace']+['format']}

def route(s) -> Literal['read','write','reply']:
    t = s['intent'].get('tool')
    if not t: return 'reply'
    return TOOLS[t][1] if TOOLS[t][1]=='write' else 'read'

print('Nodes OK')

Nodes OK


## 4. Compile Graph

In [5]:
g = StateGraph(S)
for name,fn in [('parse',parse),('read',read_tool),
                ('stage',stage_write),('exec',exec_confirmed),('fmt',fmt)]:
    g.add_node(name, fn)
g.set_entry_point('parse')
g.add_conditional_edges('parse', route, {'read':'read','write':'stage','reply':'fmt'})
g.add_edge('read','fmt'); g.add_edge('stage',END)
g.add_edge('exec','fmt'); g.add_edge('fmt',END)
agent = g.compile()
print('Compiled. Paths:')
print('  READ  -> read -> fmt -> END')
print('  WRITE -> stage -> END  [pause for confirm]')
print('  OTHER -> fmt -> END')

Compiled. Paths:
  READ  -> read -> fmt -> END
  WRITE -> stage -> END  [pause for confirm]
  OTHER -> fmt -> END


## 5. Read Queries

In [6]:
def run(msg, confirm=None):
    return agent.invoke(S(msg=msg,intent=None,result=None,
                          pending=None,confirm=confirm,answer=None,trace=[]))

for q in ['Show me pending jobs','What is the schedule overview?']:
    r = run(q)
    print(f'Q: {q}')
    print(f'A: {r["answer"]}')
    print(f'Trace: {" -> ".join(r["trace"])}\n')

Q: Show me pending jobs
A: 2 results: {'id': 'J001', 'customer': 'Smith HVAC', 'status': 'pending', 'tech': None, 'value': 850}; {'id': 'J002', 'customer': 'Jones Plumbing', 'status': 'pending', 'tech': None, 'value': 1200}
Trace: parse -> read -> format

Q: What is the schedule overview?
A: pending_jobs:2 | available_techs:2 | utilization:33%
Trace: parse -> read -> format



## 6. Write Op -- Confirm-Before-Execute

In [7]:
r = run('Assign T02 to J001')
print(f'Paused at: {r["pending"]}')

g2 = StateGraph(S)
g2.add_node('exec',exec_confirmed); g2.add_node('fmt',fmt)
g2.set_entry_point('exec')
g2.add_edge('exec','fmt'); g2.add_edge('fmt',END)
exec_g = g2.compile()

print('\n--- CONFIRM ---')
f = exec_g.invoke(S(**{**r,'confirm':True}))
print(f'Result : {f["answer"]}')

print('\n--- CANCEL ---')
f2 = exec_g.invoke(S(**{**r,'confirm':False}))
print(f'Result : {f2["answer"]}')

  CONFIRM REQUIRED: assign_tech({'job_id': 'J001', 'tech_id': 'T02'})
Paused at: {'tool': 'assign_tech', 'args': {'job_id': 'J001', 'tech_id': 'T02'}}

--- CONFIRM ---
Result : Assigned Priya S. to J001 (Smith HVAC)

--- CANCEL ---
Result : I help with scheduling and dispatch.


## 7. Streaming Simulation

In [8]:
def stream(msg):
    r = run(msg)
    print(f'User : {msg}')
    print('Atlas: ', end='', flush=True)
    for w in (r['answer'] or '').split():
        print(w+' ', end='', flush=True)
        time.sleep(0.04)
    print()

# Production: agent.astream() + Azure OpenAI stream=True
stream('Show me all available technicians')
print()
stream('What is the current schedule overview?')

User : Show me all available technicians
Atlas: 

2 

results: 

{'id': 

'T02', 

...

pending_jobs:1 

| 

available_techs:2 

| 

utilization:33% 

## 8. Design Tradeoffs

In [9]:
rows = [
    ('Rule-based routing',     'O(k) deterministic',   'Brittle on edge cases'),
    ('LLM-based routing',      'Handles ambiguity',    'Latency + cost per call'),
    ('Confirm-before-execute', 'Safe, auditable',      'Extra round-trip for writes'),
    ('Immediate execution',    'Fast UX',              'Catastrophic if write errors'),
    ('StateGraph',             'Explicit, cyclable',   'More setup vs simple chains'),
]
print(f'  {"Pattern":<28} {"Pro":<30} Con')
print('-'*75)
for r,p,c in rows:
    print(f'  {r:<28} {p:<30} {c}')
print()
print('COMPLEXITY')
print('  Intent:   O(k) rules | O(1) LLM call')
print('  Tool:     O(log n) with DB index')
print('  Graph:    O(V+E) per run -- tiny for agent graphs')
print('  Memory:   O(log n) ANN retrieval')

  Pattern                      Pro                            Con
---------------------------------------------------------------------------
  Rule-based routing           O(k) deterministic             Brittle on edge cases
  LLM-based routing            Handles ambiguity              Latency + cost per call
  Confirm-before-execute       Safe, auditable                Extra round-trip for writes
  Immediate execution          Fast UX                        Catastrophic if write errors
  StateGraph                   Explicit, cyclable             More setup vs simple chains

COMPLEXITY
  Intent:   O(k) rules | O(1) LLM call
  Tool:     O(log n) with DB index
  Graph:    O(V+E) per run -- tiny for agent graphs
  Memory:   O(log n) ANN retrieval


---
## Why This Approach? Rationale and Alternatives

### Confirm-Before-Execute: Why It's Not Optional for Production

A pure tool-calling agent that executes writes immediately has two failure modes:
1. **LLM hallucination**: The model misparses the intent ("assign T02 to J001" becomes
   "assign T01 to J002"). In a chatbot this is annoying; in a dispatch system it sends
   the wrong tech to the wrong job.
2. **No audit trail**: For billing and compliance, every tech assignment and payment
   trigger needs to know who authorized it. An autonomous LLM action has no human in the
   loop.

The confirm-before-execute pattern solves both. The graph pauses at the `stage_write`
node, shows the user what it's about to do, and only continues when the user explicitly
confirms. This adds one round-trip but makes write operations safe to deploy in production.

### Mock LLM vs. Real Azure OpenAI

The `mock_llm()` function in this notebook uses keyword matching instead of a real LLM.
In production Atlas, this node is an Azure OpenAI function-calling request:

```python
# Production version of the parse node
response = openai_client.chat.completions.create(
    model="gpt-4",
    messages=[{"role": "user", "content": state["msg"]}],
    tools=TOOL_SCHEMAS,          # JSON schema of each tool
    tool_choice="auto"
)
tool_call = response.choices[0].message.tool_calls[0]
return {"intent": {"tool": tool_call.function.name,
                   "args": json.loads(tool_call.function.arguments)}}
```

The graph structure, state transitions, and confirm-before-execute logic are identical
whether you use the mock or GPT-4. This is the right way to develop agents: build and
test the scaffolding with a cheap deterministic mock, then swap in the real LLM.


In [10]:
# ── Trace every path through the graph ────────────────────────────────────────
# One of the best ways to understand a LangGraph agent is to trace every possible
# path and verify the state at each node.

print("=" * 60)
print("PATH ANALYSIS: All routes through the Atlas state graph")
print("=" * 60)

test_cases = [
    # (message, confirm, expected_path, description)
    ("Show me pending jobs",          None,  "parse -> read -> fmt", "READ query"),
    ("What is the schedule overview?",None,  "parse -> read -> fmt", "READ query 2"),
    ("Hi, what can you do?",          None,  "parse -> fmt",         "DIRECT (no tool)"),
    ("Assign T02 to J001",            None,  "parse -> stage -> END","WRITE paused"),
    ("Assign T02 to J001 [confirmed]",True,  "exec -> fmt",          "WRITE confirmed"),
    ("Assign T02 to J001 [cancelled]",False, "exec -> fmt",          "WRITE cancelled"),
]

for msg, confirm, expected_path, description in test_cases:
    if confirm is None and "confirmed" not in msg and "cancelled" not in msg:
        result = run(msg)
    elif confirm is not None:
        # First run to get the staged state
        staged = run("Assign T02 to J001")
        from langgraph.graph import StateGraph, END
        g_exec = StateGraph(S)
        g_exec.add_node('exec', exec_confirmed)
        g_exec.add_node('fmt', fmt)
        g_exec.set_entry_point('exec')
        g_exec.add_edge('exec', 'fmt')
        g_exec.add_edge('fmt', END)
        exec_only = g_exec.compile()
        result = exec_only.invoke(S(**{**staged, 'confirm': confirm}))
    else:
        result = run(msg)

    actual_trace = " -> ".join(result.get('trace', []))
    status = "PASS" if expected_path in actual_trace else f"DIFF (got: {actual_trace})"
    print(f"\n[{status}] {description}")
    print(f"  Input:    {msg[:50]}")
    print(f"  Expected: {expected_path}")
    print(f"  Trace:    {actual_trace}")
    if result.get('answer'):
        print(f"  Answer:   {result['answer'][:80]}")
    if result.get('pending'):
        print(f"  Staged:   {result['pending']}")


PATH ANALYSIS: All routes through the Atlas state graph

[DIFF (got: parse -> read -> format)] READ query
  Input:    Show me pending jobs
  Expected: parse -> read -> fmt
  Trace:    parse -> read -> format
  Answer:   1 results: {'id': 'J002', 'customer': 'Jones Plumbing', 'status': 'pending', 'te

[DIFF (got: parse -> read -> format)] READ query 2
  Input:    What is the schedule overview?
  Expected: parse -> read -> fmt
  Trace:    parse -> read -> format
  Answer:   pending_jobs:1 | available_techs:2 | utilization:33%

[DIFF (got: parse -> format)] DIRECT (no tool)
  Input:    Hi, what can you do?
  Expected: parse -> fmt
  Trace:    parse -> format
  Answer:   I help with scheduling and dispatch.
  CONFIRM REQUIRED: assign_tech({'job_id': 'J001', 'tech_id': 'T02'})

[DIFF (got: parse -> stage)] WRITE paused
  Input:    Assign T02 to J001
  Expected: parse -> stage -> END
  Trace:    parse -> stage
  Staged:   {'tool': 'assign_tech', 'args': {'job_id': 'J001', 'tech_id': 'T02'}}


In [11]:
# ── Extending Atlas: adding a new tool ────────────────────────────────────────
# The pattern for adding capabilities to Atlas is always the same 3 steps.
# This shows how to add a "revenue report" tool.

# Step 1: Define the function
def get_revenue_report(days: int = 7) -> dict:
    # Returns total revenue and job count for the last N days
    import random
    random.seed(days)
    n_jobs   = random.randint(20, 60) * days // 7
    revenue  = sum(random.uniform(150, 2500) for _ in range(n_jobs))
    avg_job  = revenue / n_jobs
    return {
        'period_days': days,
        'total_jobs': n_jobs,
        'total_revenue': round(revenue, 2),
        'avg_job_value': round(avg_job, 2),
        'top_job_type': random.choice(['AC Install', 'Furnace Repair', 'Drain Clear']),
    }

# Step 2: Register it as a READ tool (no confirmation needed — it's a read)
TOOLS['get_revenue_report'] = (get_revenue_report, 'read')

# Step 3: Add it to the mock LLM's routing (in production: add to OpenAI tool schema)
_original_mock_llm = mock_llm
def mock_llm_v2(msg):
    m = msg.lower()
    if 'revenue' in m or 'report' in m or 'total' in m or 'earned' in m:
        days_match = __import__('re').search(r'(\d+)\s*day', m)
        days = int(days_match.group(1)) if days_match else 7
        return {'tool': 'get_revenue_report', 'args': {'days': days}}
    return _original_mock_llm(msg)  # fall through to original

# Temporarily override for demo
import builtins
_saved = mock_llm
mock_llm = mock_llm_v2  # noqa: F811

# Update the parse node to use the new mock
def parse_v2(s):
    return {'intent': mock_llm_v2(s['msg']), 'trace': s['trace'] + ['parse']}

# Test it
print("=== New Tool: Revenue Report ===\n")
sample_report = get_revenue_report(days=7)
print("Direct call:", sample_report)

print("\nMock LLM routing test:")
for q in ["Show me revenue for the last 7 days",
          "What did we earn this week?",
          "Revenue report for 30 days"]:
    intent = mock_llm_v2(q)
    print(f"  '{q}' -> {intent}")

mock_llm = _saved  # restore


=== New Tool: Revenue Report ===

Direct call: {'period_days': 7, 'total_jobs': 40, 'total_revenue': 49534.7, 'avg_job_value': 1238.37, 'top_job_type': 'Furnace Repair'}

Mock LLM routing test:
  'Show me revenue for the last 7 days' -> {'tool': 'get_revenue_report', 'args': {'days': 7}}
  'What did we earn this week?' -> {'tool': None, 'args': {}}
  'Revenue report for 30 days' -> {'tool': 'get_revenue_report', 'args': {'days': 30}}


In [12]:
# ── Production considerations: what changes when you deploy Atlas for real ─────
print("=== Atlas Production Checklist ===\n")

checklist = [
    ("LLM", "Replace mock_llm() with Azure OpenAI function_calling",
     "Use gpt-4 or gpt-4o with tool_choice='auto'"),
    ("Auth", "Add tenant_id to every tool call",
     "Atlas must only read/write data for the authenticated contractor"),
    ("Audit log", "Log every write operation with user_id + timestamp",
     "Required for compliance; enables rollback"),
    ("Rate limiting", "Cap LLM calls per user per minute",
     "Prevents runaway agent loops; cost control"),
    ("Streaming", "Use agent.astream() + SSE to frontend",
     "agent.invoke() blocks — streaming gives responsive UX"),
    ("Error handling", "Wrap tool calls in try/except; return structured errors",
     "LLM should receive 'tool failed: X' not an exception"),
    ("Confirmation UX", "Stage writes to a UI modal, not just text",
     "Text confirmation is fragile; a yes/no button is explicit"),
    ("Memory", "Add conversation history to state (list of messages)",
     "Current graph is single-turn; Atlas needs context window"),
    ("Multi-step", "Add a 'loop' edge from fmt back to parse",
     "Enables 'find a tech AND assign them' in one user message"),
]

for category, what, how in checklist:
    print(f"[{category:>15}] {what}")
    print(f"                  -> {how}\n")


=== Atlas Production Checklist ===

[            LLM] Replace mock_llm() with Azure OpenAI function_calling
                  -> Use gpt-4 or gpt-4o with tool_choice='auto'

[           Auth] Add tenant_id to every tool call
                  -> Atlas must only read/write data for the authenticated contractor

[      Audit log] Log every write operation with user_id + timestamp
                  -> Required for compliance; enables rollback

[  Rate limiting] Cap LLM calls per user per minute
                  -> Prevents runaway agent loops; cost control

[      Streaming] Use agent.astream() + SSE to frontend
                  -> agent.invoke() blocks — streaming gives responsive UX

[ Error handling] Wrap tool calls in try/except; return structured errors
                  -> LLM should receive 'tool failed: X' not an exception

[Confirmation UX] Stage writes to a UI modal, not just text
                  -> Text confirmation is fragile; a yes/no button is explicit

[         Memory]

---
## Interview Q&A — LangGraph Agents / Atlas

---

**Q: What is LangGraph and why use it instead of a simple LLM + prompt?**

A: LangGraph is a framework for building stateful, multi-step LLM workflows as explicit
directed graphs. The key advantage over a simple prompt-and-respond pattern is control
over execution flow. With a plain LLM call, you get one response. With LangGraph, you
can route to different nodes based on what the LLM decides, pause execution mid-flow to
wait for human confirmation, loop back for multi-step tasks, and maintain explicit typed
state across nodes. For Atlas, the confirm-before-execute safety requirement alone makes
a state machine necessary — there's no clean way to pause-and-resume with a simple chain.

---

**Q: Why separate tools into READ and WRITE categories?**

A: Read operations are safe to execute immediately — getting a list of pending jobs has
no side effects and can be retried if it fails. Write operations mutate real business
state: assigning a tech, triggering a payment, modifying a schedule. If the LLM
misparses the intent for a read, you get a wrong answer. If it misparses a write, you
send the wrong tech to the wrong job or charge a customer incorrectly. The READ/WRITE
split maps directly to the CQRS pattern and allows completely different safety guarantees
for each path. It also makes the system auditable: every write has a human confirmation
event in the log.

---

**Q: How does Atlas handle multi-tenant data isolation? A dispatcher at company A shouldn't see company B's jobs.**

A: Two layers. First, every tool function receives `tenant_id` from the authenticated
session and uses it as a filter on every database query. The LLM never has access to
raw database credentials — it calls tool functions, which enforce isolation internally.
Second, the LLM's context window is populated only with data for the authenticated tenant,
so it can't construct queries that leak across tenants even if it tried. Azure Service Bus
messages are also partitioned by tenant_id, so the agent's event triggers are isolated.

---

**Q: What's the difference between LangGraph and a traditional FSM (finite state machine)?**

A: A traditional FSM has fixed, enumerated states and transitions. LangGraph is a graph
where the transitions are determined at runtime by LLM outputs — the graph structure is
fixed, but which path is taken depends on what the model decides. This gives you the
safety and auditability of an FSM (you know every possible path through the graph)
combined with the flexibility of LLM-driven routing. The practical impact: you can add
a new tool by adding a node and a conditional edge, without rewriting the routing logic.

---

**Q: What happens when the LLM produces an intent that doesn't match any registered tool?**

A: The `route()` function handles this as the "reply" path — when no tool is recognized,
the graph routes to the `fmt` node which generates a direct conversational response.
In production you'd also want a fallback tool like `clarify_intent()` that asks the user
to rephrase. The important thing is that unrecognized intents never reach `exec_confirmed`
— they can't accidentally trigger a write action on a misparsed tool name.

---

**Q: How would you add memory to Atlas so it can handle "now assign that tech to the job I mentioned earlier"?**

A: Add a `history` field to the state TypedDict as a list of `(user_msg, assistant_response)`
tuples. On each parse node invocation, include the last N turns of history in the LLM
prompt alongside the current message. The LLM can then resolve coreferences ("that tech",
"the job I mentioned"). For longer conversations, use a vector store (Redis with cosine
similarity) to retrieve only the most relevant past turns rather than the full history —
this keeps the context window manageable for high-volume use.
